### Load the libraries needed

In [1]:
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

### Load the data

In [2]:
df = datasets.load_diabetes(as_frame=True).frame
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


### Information about data

In [3]:
# information about the data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     442 non-null    float64
 1   sex     442 non-null    float64
 2   bmi     442 non-null    float64
 3   bp      442 non-null    float64
 4   s1      442 non-null    float64
 5   s2      442 non-null    float64
 6   s3      442 non-null    float64
 7   s4      442 non-null    float64
 8   s5      442 non-null    float64
 9   s6      442 non-null    float64
 10  target  442 non-null    float64
dtypes: float64(11)
memory usage: 38.1 KB


In [4]:
# check number of null values
df.isnull().sum()

age       0
sex       0
bmi       0
bp        0
s1        0
s2        0
s3        0
s4        0
s5        0
s6        0
target    0
dtype: int64

### Split the data into training and testing part

In [5]:
# dependent variable
x = df.drop(columns=["target"])
x.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [6]:
# independent variable
y = df["target"]
y.head()

0    151.0
1     75.0
2    141.0
3    206.0
4    135.0
Name: target, dtype: float64

In [7]:
# train test split
train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.3, random_state = 42)

In [8]:
len(train_x), len(test_x), len(train_y), len(test_y)

(309, 133, 309, 133)

### Preprocessing data

In [9]:
# All values are scaled except target (y)
y_scaler = StandardScaler()

# covert data before transforming it to 2D and then back to 1D
train_y_scaled = y_scaler.fit_transform(train_y.values.reshape(-1, 1)).ravel()
test_y_scaled = y_scaler.transform(test_y.values.reshape(-1, 1)).ravel()

### Create and Evaluate Model

In [10]:
# create object
svr = SVR()

In [11]:
# create model
svr.fit(train_x, train_y_scaled)

SVR()

In [12]:
# predict
y_pred_scaled = svr.predict(test_x)

In [13]:
# Evaluate
print("r2 score: ", r2_score(test_y_scaled, y_pred_scaled))

r2 score:  0.48844443151651895


In [14]:
# Linear kernel

# create model
svr = SVR(kernel="linear")
svr.fit(train_x, train_y_scaled)

# predict
y_pred_scaled = svr.predict(test_x)

# Evaluate
print("r2 score: ", r2_score(test_y_scaled, y_pred_scaled))

r2 score:  0.44337613238337736


In [15]:
# poly kernel

# create model
svr = SVR(kernel="poly")
svr.fit(train_x, train_y_scaled)

# predict
y_pred_scaled = svr.predict(test_x)

# Evaluate
print("r2 score: ", r2_score(test_y_scaled, y_pred_scaled))

r2 score:  0.24203771038107835


In [16]:
# sigmoid kernel

# create model
svr = SVR(kernel="sigmoid")
svr.fit(train_x, train_y_scaled)

# predict
y_pred_scaled = svr.predict(test_x)

# Evaluate
print("r2 score: ", r2_score(test_y_scaled, y_pred_scaled))

r2 score:  -15.316808189576822


### Hyper parameter tuning using GridSearchCV

In [17]:
# import GridSearchCV
from sklearn.model_selection import GridSearchCV

# dictionary for all parameters
param_grid = {
    "C": [1, 2, 5, 10, 50, 100],
    "kernel": ["rbf", "linear", "poly"],
    "epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

In [18]:
# create model object
svr = SVR()

grid_search = GridSearchCV(svr, param_grid, scoring="r2", cv=5)
grid_search.fit(train_x, train_y_scaled)

GridSearchCV(cv=5, estimator=SVR(),
             param_grid={'C': [1, 2, 5, 10, 50, 100],
                         'epsilon': [0.01, 0.1, 0.2, 0.3, 0.5],
                         'kernel': ['rbf', 'linear', 'poly']},
             scoring='r2')

In [19]:
# check best parameters
print("best params: ", grid_search.best_params_)

best params:  {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [20]:
# create best model
best_model = SVR(kernel="linear", C=10, epsilon=0.1)
best_model.fit(train_x, train_y_scaled)

SVR(C=10, kernel='linear')

In [21]:
# predict
y_pred_scaled = best_model.predict(test_x)

# Evaluate
print("r2 score: ", r2_score(test_y_scaled, y_pred_scaled))

r2 score:  0.47444183250401084
